# Resource estimation for SOSSA sum-of-squares spectral amplification

This notebook estimates the cost of quantum phase estimation (QPE) for Hamiltonians
block-encoded with **SOSSA** (sum-of-squares spectral amplification, Low *et al.*,
[arXiv:2502.15882](https://arxiv.org/abs/2502.15882)). The Hamiltonian is expressed in a sum-of-squares form:

$$
H=\sum_{\alpha}O_{\alpha}^{\dagger}O_{\alpha}+E_{\mathrm{SOS}}
=H_{\mathrm{sqrt}}^{\dagger}H_{\mathrm{sqrt}}+E_{\mathrm{SOS}}.
$$

The square-root transformation stretches the low-energy spectrum, allowing QPE on the SOSSA walk
to use coarser phase resolution while achieving the same energy precision.

We demonstrate resource estimation through three workflows:
1. [Part 1: stored DFTHC H<sub>2</sub>](#part-1) loads a serialized double-factorized tensor hypercontraction (DFTHC) Hamiltonian from a JSON file.
2. [Part 2: DF for stretched N<sub>2</sub>](#part-2) constructs an active-space Hamiltonian and applies double factorization (DF).
3. [Part 3: synthetic examples](#part-3) provides resource estimates based on $(N, R, B, C)$ shapes.

Each part constructs a unary-iteration QPE circuit and estimates its resources using `qdk.qre`.

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md),
install the `jupyter`, `plugins`, and `qre` extras:

```bash
pip install 'qdk-chemistry[jupyter,plugins,qre]'
```

In [ ]:
from pathlib import Path

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    Configuration,
    Hamiltonian,
    MajoranaMapping,
    StateVectorContainer,
    Wavefunction,
)
from qdk_chemistry.utils import Logger
from utils.sossa_utils import (
    heisenberg_queries,
    make_fake_hamiltonian,
    sossa_unary_qpe_circuit,
)

TARGET_PRECISION = 1e-3
MAX_ERROR = 0.01
Logger.set_global_level(Logger.LogLevel.off)

<a id="part-1"></a>

## Part 1: SOSSA from a stored DFTHC Hamiltonian

Double-factorized tensor hypercontraction (DFTHC) approximates the electronic Hamiltonian with
low-rank factors that cast the two-electron interaction as a sum of squares:

$$
\begin{aligned}
H_{\mathrm{DFTHC}}
={}&\sum_{pq}h_{pq}^{(1)\prime}E_{pq}
+\frac{1}{2}\sum_{r\in[R]}\sum_{c\in[C]}\left(
W^{(rc)}\mathbb{I}
+\sum_{b=0}^{B-1}w_b^{(rc)}\sum_{pq}u_{b,p}^{(r)}u_{b,q}^{(r)}E_{pq}
\right)^2 \\
&-\frac{1}{2}\sum_{r\in[R]}\sum_{c\in[C]}\left|W^{(rc)}\right|^2,
\qquad
E_{pq}=\sum_{\sigma}a_{p\sigma}^{\dagger}a_{q\sigma}.
\end{aligned}
$$

The first example loads a serialized `FactorizedHamiltonianContainer` for H<sub>2</sub> with
$N=2$ spatial orbitals, $R=2$ ranks, $B=2$ bases, and $C=1$ copy.

In [ ]:
json_path = Path("data") / "h2_dfthc_r2_b2_c1.hamiltonian.json"
h2_hamiltonian = Hamiltonian.from_json(json_path.read_text())
h2_container = h2_hamiltonian.get_container()

print(h2_hamiltonian.get_summary())

### Build the SOSSA QPE circuit

We build the circuit one stage at a time. First, the SOS qubit mapper applies the Jordan-Wigner
transformation to the factorized Hamiltonian, producing the sum-of-squares qubit operator

$$
\begin{aligned}
H\approx H_{\mathrm{DFTHC}}
={}&\sum_{G\in\{\mathrm{D}_1,\mathrm{Q}_1\}}\sum_r\sum_{\sigma\in\{0,1\}}
O_{G^{\sigma,r}}^{\dagger}O_{G^{\sigma,r}} \\
&+\sum_{r\in[R],c\in[C]}O_{\mathrm{SF},rc}^{\dagger}O_{\mathrm{SF},rc}
+E_{\mathrm{SOS}}.
\end{aligned}
$$

In [ ]:
h2_num_orbitals = h2_container.get_num_orbitals()
h2_operator = create("qubit_mapper", "sos").run(
    h2_hamiltonian,
    MajoranaMapping.jordan_wigner(2 * h2_num_orbitals),
)

The Hamiltonian unitary builder then constructs the SOSSA walk from the mapped operator. The
Hermitian square-root construction satisfies

$$
H_{\mathrm{sqrt}}^{\prime\,2}=
\begin{pmatrix}
H_{\mathrm{SA}} & \mathbf{0} \\
\mathbf{0} & \ddots
\end{pmatrix},
$$

where the upper diagonal block is the original gap-amplifiable Hamiltonian $H_{\mathrm{SA}}$. The block encoding yields

$$
U:=\mathrm{SEL}\cdot\mathrm{PREP}^{\dagger}=
\mathrm{BE}\!\left[\frac{H_{\mathrm{sqrt}}^{\prime}}{\lambda_{\mathrm{sqrt}}}\right],
$$

where
$$
\mathrm{SEL}=
\sum_{\alpha}|\alpha\rangle\langle\alpha|\otimes
\mathrm{BE}\!\left[O_{\alpha}/\lambda_{\alpha}\right],
$$

$$
\begin{aligned}
\mathrm{PREP}|0\rangle_{\mathrm a}
&=\sum_{\alpha}\frac{\lambda_{\alpha}}{\lambda_{\mathrm{sqrt}}}
|\chi_{\alpha}\rangle_{\mathrm a},
\qquad
\lambda_{\mathrm{sqrt}}=\sqrt{\sum_{\alpha}\lambda_{\alpha}^{2}}.
\end{aligned}
$$

and the reflected walk satisfies

$$
\begin{aligned}
\frac{2H_{\mathrm{SA}}}{\lambda_{\mathrm{sqrt}}^{2}}-\mathbb{I}
&=\langle 0|_{\mathrm a\mathcal B}U^{\dagger}\mathrm{REF}_{\mathcal B}
U|0\rangle_{\mathrm a\mathcal B}.
\end{aligned}
$$

In [ ]:
h2_walk = create("hamiltonian_unitary_builder", "sossa").run(h2_operator)
h2_walk_container = h2_walk.get_container()

We then configure the circuit mapper for SOSSA. The outer PREPARE selects $\alpha$, the inner
PREPARE loads the terms of $O_\alpha$, SELECT applies those terms, and reflections form the
rectangular SOSSA walk used by QPE. These algorithmic primitives are modular, so we can customize
them to explore design choices in state preparation, table lookup, and rotations.

In [ ]:
h2_circuit_mapper = AlgorithmRef(
    "circuit_mapper",
    "sossa",
    outer_prepare=AlgorithmRef("state_prep", "dense_pure_state"),
    inner_prepare_algorithm="direct",
    select_algorithm="direct",
)

We prepare the molecule's Hartree-Fock reference state.

In [ ]:
h2_hf_configuration = Configuration.canonical_hf_configuration(
    1,
    1,
    h2_num_orbitals,
)
h2_reference = Wavefunction(
    StateVectorContainer(h2_hf_configuration, h2_container.get_orbitals())
)
h2_state_preparation = create("state_prep", "sparse_isometry").run(h2_reference)

Unary-iteration QPE prepares a time register over $p+1$ slots and applies $p$ SOSSA walk queries
using unary iteration. Unlike a binary-power schedule, $p$ can be any positive integer and does not
need to be rounded up to a power of two. For a target energy precision $\sigma_E$, the
Heisenberg-limited query count is

$$
p=\left\lceil\frac{\pi\lambda_{\mathrm{eff}}}{2\sigma_E}\right\rceil,
\qquad
\lambda_{\mathrm{eff}}=\sqrt{E_{\mathrm{gap}}(2\Lambda-E_{\mathrm{gap}})}.
$$

Near the low-energy band edge, $\lambda_{\mathrm{eff}}<\Lambda$ reflects the coarser phase resolution
enabled by the square-root transformation. Part 1 does not include a classical reference energy from
which to obtain $E_{\mathrm{gap}}$, so we conservatively substitute $\Lambda$ for
$\lambda_{\mathrm{eff}}$.

In [ ]:
h2_queries = heisenberg_queries(
    h2_walk_container.normalization,
    TARGET_PRECISION,
)

h2_qpe_builder = create(
    "qpe_circuit_builder",
    "qdk_unary",
    num_queries=h2_queries,
    circuit_mapper=h2_circuit_mapper,
    unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa"),
)

h2_circuit = h2_qpe_builder.run(
    state_preparation=h2_state_preparation,
    qubit_hamiltonian=h2_operator,
)[0]

### Validate the SOSSA circuit on H<sub>2</sub>

A small factorized H<sub>2</sub> Hamiltonian can be simulated directly, allowing us to verify that
QPE recovers the classically computed energy before estimating resources.

Constructing the circuit explicitly also validates the resource-estimation input and provides a
baseline for future improvements.

In [ ]:
# This cell takes ~30 seconds to run.
from qdk.widgets import Histogram

SIMULATION_QUERIES = 15
SIMULATION_SHOTS = 50
SIMULATION_SEED = 42

h2_simulation_qpe_builder = create(
    "qpe_circuit_builder",
    "qdk_unary",
    num_queries=SIMULATION_QUERIES,
    circuit_mapper=h2_circuit_mapper,
    unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa"),
)
h2_circuit = h2_simulation_qpe_builder.run(
    state_preparation=h2_state_preparation,
    qubit_hamiltonian=h2_operator,
)[0]

h2_execution = create("circuit_executor", "qdk_sparse_state_simulator", seed=SIMULATION_SEED).run(
    h2_circuit,
    shots=SIMULATION_SHOTS,
)
h2_phase_probabilities = {
    bitstring: count / h2_execution.total_shots
    for bitstring, count in h2_execution.bitstring_counts.items()
}
display(Histogram(bar_values=h2_phase_probabilities))

The simulation recovers the two expected conjugate phase peaks.

### Physical resource estimates

`qdk.qre` maps the logical circuit onto a fault-tolerant architecture and returns
Pareto-optimal trade-offs between physical qubit count and runtime. We use a Majorana-based
architecture with a $10^{-5}$ physical error rate, the `ThreeAux` code, and round-based magic-state
factories. We set the total error budget to 1%.

In [ ]:
from qdk.qre import estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

architecture = Majorana(error_rate=1e-5)
isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())

h2_results = estimate(
    h2_circuit.get_qre_application(), architecture, isa_query, max_error=0.01, name="SOSSA_H2"
)
h2_results.add_factory_summary_column()
display(h2_results.as_frame())

plot_estimates(h2_results, figsize=(6, 4), runtime_unit="ms")

<a id="part-2"></a>

## Part 2: Double factorization of stretched N<sub>2</sub>

We condense the classical preparation here to keep the focus on SOSSA resource estimation. See the
[stretched N<sub>2</sub> QPE notebook](qpe_stretched_n2.ipynb) for the complete workflow and discussion.

In [ ]:
from qdk_chemistry.data import Structure
from qdk_chemistry.data.symmetry import SymmetryLabel, axes
from qdk_chemistry.utils import compute_valence_space_parameters

structure = Structure.from_xyz_file(Path("data/stretched_n2.structure.xyz"))
scf_solver = create("scf_solver")
e_hf, wfn_hf = scf_solver.run(
    structure,
    charge=0,
    spin_multiplicity=1,
    basis_or_guess="cc-pvdz",
)

num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
active_space_selector = create(
    "active_space_selector",
    "qdk_valence",
    num_active_electrons=num_val_e,
    num_active_orbitals=num_val_o,
)
valence_wfn = active_space_selector.run(wfn_hf)

localizer = create("orbital_localizer", "qdk_mp2_natural_orbitals")
valence_indices = valence_wfn.get_orbitals().active_indices()
localized_wfn = localizer.run(
    valence_wfn,
    list(valence_indices.indices(SymmetryLabel([axes.alpha()]))),
    list(valence_indices.indices(SymmetryLabel([axes.beta()]))),
)

hamiltonian_constructor = create("hamiltonian_constructor")
localized_hamiltonian = hamiltonian_constructor.run(localized_wfn.get_orbitals())
num_alpha_electrons, num_beta_electrons = localized_wfn.get_active_num_electrons()
selected_ci = create(
    "multi_configuration_calculator",
    "macis_asci",
    calculate_one_rdm=True,
    calculate_two_rdm=True,
)
_, selected_ci_wfn = selected_ci.run(localized_hamiltonian, num_alpha_electrons, num_beta_electrons)

autocas = create("active_space_selector", "qdk_autocas_eos")
autocas_wfn = autocas.run(selected_ci_wfn)
active_hamiltonian = hamiltonian_constructor.run(autocas_wfn.get_orbitals())
alpha_electrons, beta_electrons = autocas_wfn.get_active_num_electrons()

casci = create("multi_configuration_calculator", "macis_cas")
e_cas, _ = casci.run(active_hamiltonian, alpha_electrons, beta_electrons)
print(f"Hartree-Fock energy: {e_hf:.6f} Hartree")
print(f"Active-space CASCI energy: {e_cas:.6f} Hartree")
print(f"Active space: {alpha_electrons} alpha + {beta_electrons} beta electrons")

### Double factorization

In the general DFTHC representation, conventional DF is the special case $R=R_{\mathrm{DF}}$, $B=N$, and
$C=1$, in which each two-body factor is diagonalized:

$$
g_{pqrs}\approx\sum_{Q=0}^{R_{\mathrm{DF}}-1}L_{pq}^{(Q)}L_{rs}^{(Q)},
\qquad
L_{pq}^{(Q)}=\sum_{b=0}^{N-1}w_b^{(Q)}u_{b,p}^{(Q)}u_{b,q}^{(Q)}.
$$

In [ ]:
factorizer = create("double_factorizer", "qdk")
factorizer.settings().set("truncation_threshold", 1e-8)
n2_hamiltonian = factorizer.run(active_hamiltonian)
n2_container = n2_hamiltonian.get_container()

print(n2_hamiltonian.get_summary())

### Physical resource estimates

The following cell builds the SOSSA QPE circuit and estimates its physical resources.

In [ ]:
n2_operator = create("qubit_mapper", "sos").run(
    n2_hamiltonian, MajoranaMapping.jordan_wigner(2 * n2_container.get_num_orbitals())
)
n2_walk = (
    create("hamiltonian_unitary_builder", "sossa", reference_ground_state_energy=e_cas)
    .run(n2_operator)
    .get_container()
)

n2_queries = heisenberg_queries(n2_walk.lambda_eff, TARGET_PRECISION)

n2_circuit, _ = sossa_unary_qpe_circuit(
    n2_hamiltonian,
    num_queries=n2_queries,
    n_alpha=alpha_electrons,
    n_beta=beta_electrons,
)
n2_results = estimate(
    n2_circuit.get_qre_application(), architecture, isa_query, max_error=0.01, name="SOSSA_N2"
)
n2_results.add_factory_summary_column()
display(n2_results.as_frame())

plot_estimates(n2_results, figsize=(6, 4), runtime_unit="ms")

<a id="part-3"></a>

## Part 3: SOSSA for molecular-scale synthetic examples

This section applies the SOSSA resource-estimation procedure to synthetic molecular examples. Each
example specifies the dimensions of the factorized Hamiltonian:

- $N$: number of spatial orbitals,
- $R$: number of factorization ranks,
- $B$: number of basis vectors per rank, and
- $C$: number of coefficient copies per basis vector.

Each example also specifies the effective normalization $\lambda_{\mathrm{eff}}$ and the rotation and
coefficient precisions $b_{\mathrm{rot}}$ and $b_{\mathrm{coeff}}$. We use a target energy precision
of $\sigma_E$.

The deterministic synthetic tensors have the requested $(N, R, B, C)$ dimensions, so the estimates
approximate the resource costs of the named molecular active spaces.

In [ ]:
MOLECULES = {
    "Fe2S2 (30e, 20o)": {
        "electrons": 30,
        "N": 20,
        "R": 14,
        "B": 15,
        "C": 5,
        "lambda_eff": 6.4690,
        "b_coeff": 11,
        "b_rot": 15,
    },
    "FeMoCo (54e, 54o)": {
        "electrons": 54,
        "N": 54,
        "R": 10,
        "B": 27,
        "C": 27,
        "lambda_eff": 21.3674,
        "b_coeff": 9,
        "b_rot": 16,
    },
}

### Physical resource estimates

The following cell uses an automatic memory/compute architecture. It fixes the number of compute
qubits at 20% of the total qubit count and assigns the remaining qubits to memory.

Memory qubits store quantum states but do not support gate operations. This restriction allows them
to use more efficient error-correction codes, such as yoked surface codes.

In [ ]:
# This cell takes ~2 minutes to run.
from qdk.qre import (
    PSSPC,
    DynamicMemoryCompute,
    LatticeSurgery,
    estimate,
    plot_estimates,
)
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

MEMORY_COMPUTE_PERCENTAGE = 0.2

architecture = Majorana(error_rate=1e-5)
isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())
trace_query = DynamicMemoryCompute.q(compute_capacity_percentage=MEMORY_COMPUTE_PERCENTAGE) * PSSPC.q() * LatticeSurgery.q()

part3_results = []
for molecule, params in MOLECULES.items():
    num_electrons = int(params["electrons"])
    num_queries = heisenberg_queries(params["lambda_eff"], TARGET_PRECISION)
    hamiltonian = make_fake_hamiltonian(
        params["N"], params["R"], params["B"], params["C"]
    )
    circuit, _ = sossa_unary_qpe_circuit(
        hamiltonian,
        num_queries=num_queries,
        n_alpha=(num_electrons + 1) // 2,
        n_beta=num_electrons // 2,
        rotation_bit_precision=params["b_rot"],
        coefficient_bit_precision=params["b_coeff"],
    )
    results = estimate(
        circuit.get_qre_application(),
        architecture,
        isa_query,
        trace_query,
        max_error=MAX_ERROR,
        name=f"SOSSA {molecule} ({MEMORY_COMPUTE_PERCENTAGE}% compute)",
    )
    part3_results.append(results)
    display(results.as_frame())

plot_estimates(part3_results, figsize=(8, 5))